# Atlanta to Birmingham two-branch TCN: results report

## Executive result

An Atlanta-trained weather-residual TCN was transferred to Birmingham and evaluated on the completely held-out 2023 period. Birmingham's own structural baseline was modeled separately, then combined with the transferred residual prediction.

| Metric | Birmingham 2023 result |
|---|---:|
| MAE | **12.60 MW** |
| RMSE | **17.18 MW** |
| MAPE | **1.01%** |
| R² | **0.9955** |
| Forecasts | **8,760 hourly observations** |

## Objective

Atlanta and Birmingham have different normal demand levels and structural patterns. Predicting raw MW directly across cities would mix those differences with the effect of weather. The model therefore separates demand into a city-specific normal level and a transferable short-term adjustment:

```text
log(predicted demand at t+1)
  = log(city-specific baseline at t+1)
  + predicted weather/residual effect at t+1
```

The combined log prediction is exponentiated to produce final MW demand.

## Experiment design

| Stage | Period | Role |
|---|---|---|
| Training | 1 Jan 2016 – 31 Dec 2021 | Fit Atlanta residual TCN and the two city baseline MLPs |
| Validation | Calendar year 2022 | Select TCN epoch and tune learning rate through validation loss |
| Test | Calendar year 2023 | Final Atlanta-to-Birmingham cross-city evaluation |
| Forecast horizon | 1 hour ahead | Predict the next hourly load |
| Lookback | 168 hours | Previous 7 days of inputs for each TCN prediction |

No Birmingham 2023 target values were used to fit a baseline model, a scaler, or the Atlanta TCN. The saved notebook also verifies this chronology before reporting metrics.

### More weight for recent years

The eight-year dataset may contain demand drift from infrastructure, population, and consumption changes. Training examples are weighted with an exponential two-year half-life:

```text
weight = exp(-ln(2) × age in years / 2)
```

An observation two years older than the latest training point therefore receives half the training weight. This weighting is applied to both the baseline MLP and the Atlanta TCN training data.

## Climate-informed weather selection

The Atlanta–Birmingham similarity analysis classified the pair as moderately similar (overall score: 62.49/100). The TCN deliberately uses the variables with the strongest cross-city similarity:

| Retained core weather variable | Similarity score |
|---|---:|
| Temperature | 98.40 / 100 |
| Precipitation | 94.70 / 100 |
| Dew point | 93.18 / 100 |
| Relative humidity | 85.10 / 100 |

Pressure and visibility were excluded because their similarity was weak (8.73 and 46.35 respectively). The model additionally derives cooling-degree temperature, heating-degree temperature, and a temperature × humidity interaction.

## Data quality and leakage controls

- The two master CSVs were restored to a continuous hourly timeline.
- Invalid weather readings were changed to missing values using physical bounds.
- Weather gaps use forward-fill only, never a future observation.
- Missing load targets are never fabricated; any sequence affected by an unavailable target/input is omitted.
- Calendar splits are chronological; no random train/test split is used.
- TCN scalers are fitted on Atlanta training data only and then applied unchanged to Birmingham.
- Every 2023 Birmingham target is after the 2022 validation cutoff.

## Layer 1: city-specific structural baseline MLP

Atlanta and Birmingham each have an independently trained baseline model. The baseline learns normal city demand without weather, using calendar and long-run trend features only.

### Baseline inputs

- Hour-of-day sine and cosine
- Day-of-week sine and cosine
- Month sine and cosine
- Weekend flag
- Continuous years-since-start trend

### Architecture

```text
8 calendar/trend features
  → Dense(64, ReLU)
  → Dropout(0.10)
  → Dense(32, ReLU)
  → Dense(1): scaled log baseline demand
```

| Model | Parameters |
|---|---:|
| Atlanta baseline MLP | 2,689 |
| Birmingham baseline MLP | 2,689 |

For every hour, the baseline produces expected log demand. The model residual is then:

```text
log residual = log(actual load) − log(predicted city baseline)
```

## Layer 2: Atlanta weather-residual TCN

The TCN is trained only on Atlanta data. Its target is the next-hour log residual rather than raw MW, which makes the learned response more transferable to Birmingham.

### TCN input

Each sample has shape:

```text
(batch, 168 hours, 13 features)
```

The 13 features comprise seven weather-derived variables and six causal residual-history variables.

**Weather-derived features (7)**

- Temperature, dew point, relative humidity, hourly precipitation
- Cooling-degree and heating-degree temperature
- Temperature × humidity interaction

**Causal residual history (6)**

- Current observed residual and 1-hour, 24-hour, and 168-hour lags
- Previous 24-hour residual mean and standard deviation

The prior-residual inputs are available at prediction time because this is a one-hour-ahead forecast. Consequently, the result represents **weather plus autoregressive residual transfer**, not weather alone.

### TCN architecture and operation

```text
Input: 168 × 13
  → Residual TCN block, dilation 1
  → Residual TCN block, dilation 2
  → Residual TCN block, dilation 4
  → Residual TCN block, dilation 8
  → Residual TCN block, dilation 16
  → Residual TCN block, dilation 32
  → Residual TCN block, dilation 64
  → GlobalAveragePooling1D
  → Dense(64, ReLU)
  → Dropout(0.20)
  → Dense(1): next-hour scaled log residual
```

Every TCN block contains two causal `Conv1D` layers with 64 filters, kernel size 3, and `SpatialDropout1D(0.10)`, followed by a residual skip connection and ReLU activation. Causal convolution prevents the model from seeing later hours, while increasing dilation lets it capture both immediate and multi-day weather effects.

| TCN property | Value |
|---|---:|
| Residual blocks | 7 |
| Dilation rates | 1, 2, 4, 8, 16, 32, 64 |
| Filters per convolution | 64 |
| Kernel size | 3 |
| Input shape | 168 × 13 |
| TCN trainable parameters | **168,257** |

## GPU training and model selection

The notebook checks TensorFlow, CUDA/GPU visibility, and executes a GPU matrix-multiplication probe before training. The model is explicitly constructed on GPU:0; CPU is used only for normal file/data-pipeline operations.

The Atlanta TCN used Adam optimization with initial learning rate 0.001, mean-squared-error loss on scaled residuals, validation-based learning-rate reduction, early stopping, and best-validation checkpointing.

Training stopped after 41 epochs. The best saved validation checkpoint was epoch 29:

| Best validation checkpoint measure | Value |
|---|---:|
| Training loss | 0.01397 |
| Training MAE, scaled residual | 0.13639 |
| Validation loss | 0.02346 |
| Validation MAE, scaled residual | 0.11065 |
| Learning rate | 0.00050 |

## Cross-city reconstruction

For each Birmingham hour in 2023:

1. The Birmingham baseline MLP estimates Birmingham's normal log demand from calendar and trend inputs.
2. The Atlanta-trained TCN receives Birmingham's selected weather history and observed prior residual history.
3. The TCN predicts Birmingham's next-hour log residual using the weather/residual relationship learned in Atlanta.
4. The predicted residual is added to the Birmingham baseline in log space.
5. The result is exponentiated to obtain predicted Birmingham MW demand.

## Final interpretation

The final 1.01% MAPE and 0.9955 R² demonstrate very accurate one-hour-ahead transfer under this experiment design. The defensible conclusion is that the Atlanta-trained weather/residual relationship transferred successfully when paired with Birmingham's separately calibrated structural baseline.

This should not be described as weather-only prediction: Birmingham's own calendar/trend baseline and previously observed load residuals provide essential city-specific context. The cross-city component is the TCN relationship learned from Atlanta only.

## Saved reproducibility artifacts

- `atlanta/baseline_mlp.keras` and `birmingham/baseline_mlp.keras`
- Baseline feature and target scalers for both cities
- `atlanta_to_birmingham/best_atlanta_weather_tcn.keras`
- `atlanta_to_birmingham/final_atlanta_weather_tcn.keras`
- Atlanta TCN feature/residual scalers
- `atlanta_weather_tcn_history.csv`
- `experiment_config.json`
- `birmingham_2023_cross_city_predictions.csv`
- `birmingham_2023_cross_city_metrics.json`


# Atlanta vs Birmingham Climate Similarity Report

## Executive Summary

Atlanta and Birmingham were evaluated using a data-driven meteorological similarity procedure. The purpose is to determine whether they form a defensible similar-climate pair for the cross-city LSTM experiment.

**Final climate similarity score: 62.49/100**

**Classification: Moderately similar**

Energy consumption was not used to define climate similarity.

## 1. Data

Atlanta: `D:\Downloads\BTP\model_training\lstm_training_2\Atlanta\atlanta_master.csv`

Birmingham: `D:\Downloads\BTP\model_training\lstm_training_2\Birmingham\birmingham_master.csv`

Common observed period: **2016-01-01 00:00:00 to 2023-12-31 23:00:00**

## 2. Common Weather Variables

temperature, dew_point, relative_humidity, pressure, wind_speed, wind_direction, visibility, precipitation

## 3. Methodology

1. Detect and normalize timestamps.
2. Remove duplicate timestamps.
3. Restrict both cities to their common observed period.
4. Identify weather variables available in both datasets.
5. Compare descriptive statistics.
6. Compare empirical weather distributions using standardized mean difference, Wasserstein distance and KS statistic.
7. Compare monthly and seasonal behavior.
8. Compare weather correlation structures.
9. Standardize weather variables and calculate multivariate climate distance.
10. Combine independent similarity measures into a transparent 0–100 project score.

## 4. Variable-Level Similarity

| feature           |   Atlanta_mean |   Birmingham_mean |   Atlanta_std |   Birmingham_std |        SMD |   Wasserstein_distance |   normalized_Wasserstein |   KS_statistic |   KS_p_value |   SMD_similarity |   Wasserstein_similarity |   KS_similarity |   variable_similarity_score |
|:------------------|---------------:|------------------:|--------------:|-----------------:|-----------:|-----------------------:|-------------------------:|---------------:|-------------:|-----------------:|-------------------------:|----------------:|----------------------------:|
| temperature       |       16.7739  |           16.7063 |      9.04635  |         9.39954  | 0.00732788 |               0.258867 |                0.0280626 |      0.0130565 |  1.28439e-05 |         99.2699  |                 97.2328  |         98.6943 |                    98.399   |
| precipitation     |       22.9466  |           22.7978 |     12.1912   |        10.7935   | 0.0129302  |               1.10087  |                0.0956147 |      0.0549372 |  2.32867e-92 |         98.7153  |                 90.8814  |         94.5063 |                    94.701   |
| dew_point         |        9.52828 |           10.3371 |      9.55336  |         9.40924  | 0.0853031  |               0.811201 |                0.0855556 |      0.0407677 |  4.86553e-51 |         91.8234  |                 91.8002  |         95.9232 |                    93.1823  |
| relative_humidity |       65.955   |           69.8845 |     20.505    |        20.7641   | 0.190431   |               3.92959  |                0.190434  |      0.100171  |  1.5576e-306 |         82.6602  |                 82.66    |         89.9829 |                    85.101   |
| wind_direction    |      200.339   |          145.29   |    104.521    |       119.258    | 0.490933   |              57.3909   |                0.511818  |      0.258961  |  0           |         61.2055  |                 59.9405  |         74.1039 |                    65.0833  |
| wind_speed        |      200.339   |          145.29   |    104.521    |       119.258    | 0.490933   |              57.3909   |                0.511818  |      0.258961  |  0           |         61.2055  |                 59.9405  |         74.1039 |                    65.0833  |
| visibility        |       18.8014  |           11.6723 |     10.1335   |         4.40516  | 0.912437   |               7.25675  |                0.928779  |      0.406193  |  0           |         40.1544  |                 39.5036  |         59.3807 |                    46.3462  |
| pressure          |       98.1292  |           99.5945 |      0.514528 |         0.532616 | 2.79814    |               1.46525  |                2.79814   |      0.859989  |  0           |          6.09231 |                  6.09231 |         14.0011 |                     8.72859 |

## 5. Overall Scores

| component                        |   score_0_to_100 |   weight |
|:---------------------------------|-----------------:|---------:|
| Distribution similarity          |          69.5781 |      0.6 |
| Correlation structure similarity |          89.6078 |      0.2 |
| Multivariate climate similarity  |          14.1108 |      0.2 |
| Overall climate similarity       |          62.4906 |      1   |

## 6. Main Findings

1. 8 common meteorological variables were available: temperature, dew_point, relative_humidity, pressure, wind_speed, wind_direction, visibility, precipitation.
2. The common comparison period was 2016-01-01 to 2023-12-31.
3. Average variable-level distribution similarity was 69.58/100.
4. Weather correlation-structure similarity was 89.61/100.
5. Standardized multivariate climate distance was 1.9582.
6. Multivariate similarity was 14.11/100.
7. Overall weighted climate similarity was 62.49/100, classified as Moderately similar.
8. The most similar variable was temperature (98.40/100).
9. The least similar variable was pressure (8.73/100).

## 7. Why this supports the LSTM experiment

The city pair was not selected simply because the cities are geographically close or are generally described as having similar climates.

Instead, the pairing is evaluated in the **observed meteorological feature space**. If the weather distributions, seasonal behavior, correlation structure and standardized multivariate climate position are sufficiently similar, the pair provides a meaningful test of whether an LSTM trained using one city's weather-demand relationship can generalize to another city.

The experimental logic is:

**Weather data → climate similarity analysis → city-pair selection → LSTM training → cross-city evaluation**

## 8. Professor / Viva Explanation

> We did not select Atlanta and Birmingham arbitrarily. We first identified the weather variables available in both datasets and restricted the comparison to their common observed period. We compared their descriptive statistics and empirical distributions using standardized mean difference, Wasserstein distance and the KS statistic. We also compared the correlation structure between the meteorological variables. Finally, we standardized the common weather variables and calculated a multivariate climate distance. These independent measurements were combined into a transparent similarity score of 62.49 out of 100, which gives a classification of Moderately similar. Therefore, the city pair is justified using observed meteorological similarity rather than relying only on a general external claim that the cities have similar climates.

## 9. Limitation

Climate similarity does not imply identical energy-demand behavior. Electricity consumption is also influenced by building stock, population, commercial activity, HVAC penetration, utility infrastructure and local operating practices.

Therefore, the similarity analysis justifies the **climate pairing**, while the LSTM experiment tests whether similar meteorological regimes actually result in transferable energy-demand relationships.

## 10. Output Locations

Tables: `D:\Downloads\BTP\model_training\lstm_training_2\Atlanta\atlanta_birmingham_climate_similarity\tables`

Plots: `D:\Downloads\BTP\model_training\lstm_training_2\Atlanta\atlanta_birmingham_climate_similarity\plots`

Markdown report: `D:\Downloads\BTP\model_training\lstm_training_2\Atlanta\atlanta_birmingham_climate_similarity\report\atlanta_birmingham_climate_similarity_report.md`

HTML report: `D:\Downloads\BTP\model_training\lstm_training_2\Atlanta\atlanta_birmingham_climate_similarity\report\atlanta_birmingham_climate_similarity_report.html`
